In [28]:
import numpy as np
import sklearn.metrics
import sklearn.model_selection
import tensorflow as tf


In [29]:
MAX_TOKENS = 1000
MAX_SEQUENCE_LENGTH = 20
EMBEDDING_DIM = 16
LSTM_UNITS = 16
BATCH_SIZE = 8
EPOCHS = 30
N_SPLITS = 3

tf.keras.utils.set_random_seed(42)


In [30]:


texts = np.array([
    "claim your free prize now",
    "get rich quickly with this secret",
    "urgent account update required",
    "make money fast from home",
    "exclusive deal limited time offer",
    "meeting agenda for today",
    "project status update report",
    "lunch schedule for tomorrow",
    "team meeting reminder at office",
    "request for document review",
])

labels = np.array([
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    1,
    1,
    1,
])

print("전체 데이터 개수:", len(texts))
print("스팸메일 개수:", np.sum(labels == 0))
print("정상메일(ham) 개수:", np.sum(labels == 1))


전체 데이터 개수: 10
스팸메일 개수: 5
정상메일(ham) 개수: 5


In [31]:
train_pool_texts, test_texts, train_pool_labels, test_labels = sklearn.model_selection.train_test_split(
    texts,
    labels,
    test_size=0.15,
    random_state=42,
    stratify=labels,
)

print("train pool:", len(train_pool_texts))
print("test:", len(test_texts))

print("train pool spam:", np.sum(train_pool_labels == 0))
print("train pool ham:", np.sum(train_pool_labels == 1))
print("test spam:", np.sum(test_labels == 0))
print("test ham:", np.sum(test_labels == 1))


train pool: 8
test: 2
train pool spam: 4
train pool ham: 4
test spam: 1
test ham: 1


In [32]:
def make_dataset(texts, labels, shuffle=False):
    dataset = tf.data.Dataset.from_tensor_slices((texts, labels))

    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(texts), seed=42)

    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


In [33]:
def create_model(vectorizer):
    model = tf.keras.Sequential([
        vectorizer,
        tf.keras.layers.Embedding(
            input_dim=MAX_TOKENS,
            output_dim=EMBEDDING_DIM,
            mask_zero=True,
        ),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(LSTM_UNITS)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    return model


In [34]:
skf = sklearn.model_selection.StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=42,
)

fold_losses = []
fold_accuracies = []
best_epochs = []

for fold, (train_indices, valid_indices) in enumerate(
    skf.split(train_pool_texts, train_pool_labels),
    start=1,
):
    fold_train_texts = train_pool_texts[train_indices]
    fold_valid_texts = train_pool_texts[valid_indices]

    fold_train_labels = train_pool_labels[train_indices]
    fold_valid_labels = train_pool_labels[valid_indices]

    vectorizer = tf.keras.layers.TextVectorization(
        max_tokens=MAX_TOKENS,
        output_mode="int",
        output_sequence_length=MAX_SEQUENCE_LENGTH,
    )

    vectorizer.adapt(fold_train_texts)

    train_dataset = make_dataset(fold_train_texts, fold_train_labels, shuffle=True)
    valid_dataset = make_dataset(fold_valid_texts, fold_valid_labels)

    model = create_model(vectorizer)

    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
    )

    history = model.fit(
        train_dataset,
        validation_data=valid_dataset,
        epochs=EPOCHS,
        callbacks=[early_stopping],
        verbose=0,
    )

    valid_loss, valid_accuracy = model.evaluate(valid_dataset, verbose=0)

    fold_losses.append(valid_loss)
    fold_accuracies.append(valid_accuracy)
    best_epochs.append(len(history.history["loss"]))

    print(f"Fold {fold}")
    print(f"  validation loss: {valid_loss:.4f}")
    print(f"  validation accuracy: {valid_accuracy:.4f}")
    print(f"  trained epochs: {len(history.history['loss'])}")


Fold 1
  validation loss: 0.6926
  validation accuracy: 0.6667
  trained epochs: 6
Fold 2
  validation loss: 0.6955
  validation accuracy: 0.3333
  trained epochs: 6
Fold 3
  validation loss: 0.6936
  validation accuracy: 0.5000
  trained epochs: 10


In [35]:
print("Cross Validation Result")
print(f"mean validation loss: {np.mean(fold_losses):.4f}")
print(f"mean validation accuracy: {np.mean(fold_accuracies):.4f}")
print(f"std validation accuracy: {np.std(fold_accuracies):.4f}")
print(f"average trained epochs: {np.mean(best_epochs):.2f}")


Cross Validation Result
mean validation loss: 0.6939
mean validation accuracy: 0.5000
std validation accuracy: 0.1361
average trained epochs: 7.33


In [36]:
final_epochs = max(1, int(round(np.mean(best_epochs))))

final_vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=MAX_SEQUENCE_LENGTH,
)

final_vectorizer.adapt(train_pool_texts)

final_train_dataset = make_dataset(train_pool_texts, train_pool_labels, shuffle=True)
test_dataset = make_dataset(test_texts, test_labels)

final_model = create_model(final_vectorizer)

final_model.fit(
    final_train_dataset,
    epochs=final_epochs,
    verbose=0,
)

test_loss, test_accuracy = final_model.evaluate(test_dataset, verbose=0)

print(f"final epochs: {final_epochs}")
print(f"test loss: {test_loss:.4f}")
print(f"test accuracy: {test_accuracy:.4f}")


final epochs: 7
test loss: 0.6929
test accuracy: 0.5000


In [37]:
test_input = tf.constant(test_texts, dtype=tf.string)

test_probabilities = final_model.predict(test_input, verbose=0).flatten()
test_predictions = (test_probabilities >= 0.5).astype(int)

for text, label, probability, prediction in zip(
    test_texts,
    test_labels,
    test_probabilities,
    test_predictions,
):
    actual = "스팸메일" if label == 0 else "정상메일(ham)"
    predicted = "스팸메일" if prediction == 0 else "정상메일(ham)"

    print(text)
    print(f"  actual: {actual}")
    print(f"  spam probability: {probability:.4f}")
    print(f"  predicted: {predicted}")


make money fast from home
  actual: 스팸메일
  spam probability: 0.5006
  predicted: 정상메일(ham)
request for document review
  actual: 정상메일(ham)
  spam probability: 0.5009
  predicted: 정상메일(ham)


In [38]:
print(
    sklearn.metrics.classification_report(
        test_labels,
        test_predictions,
        target_names=["정상메일(ham)", "스팸메일"],
    )
)


              precision    recall  f1-score   support

   정상메일(ham)       0.00      0.00      0.00         1
        스팸메일       0.50      1.00      0.67         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [39]:
def predict_mail_type(texts):
    input_texts = tf.constant(texts, dtype=tf.string)
    probabilities = final_model.predict(input_texts, verbose=0).flatten()

    for text, probability in zip(texts, probabilities):
        prediction = "스팸메일" if probability >= 0.5 else "정상메일(ham)"

        print(text)
        print(f"  spam probability: {probability:.4f}")
        print(f"  prediction: {prediction}")

In [40]:
sample_texts = [
    "claim your free prize",
    "meeting agenda for tomorrow",
    "cash bonus waiting for you",
    "project document review request",
]

predict_mail_type(sample_texts)


claim your free prize
  spam probability: 0.4957
  prediction: 정상메일(ham)
meeting agenda for tomorrow
  spam probability: 0.5018
  prediction: 스팸메일
cash bonus waiting for you
  spam probability: 0.5014
  prediction: 스팸메일
project document review request
  spam probability: 0.5016
  prediction: 스팸메일
